# Agent with Retrieval
> Combine Agent + ContextPipeline + retriever for agentic RAG.

This notebook shows how to wire up an Agent with a retriever so
relevant documents are automatically injected into the context
window before every API call.

**Time:** ~7 minutes

## Setup

In [ ]:
from anchor.agent import Agent, tool, Skill, SkillRegistry
from anchor.memory import SlidingWindowMemory
from anchor.models import ContextItem, SourceType
from anchor.pipeline import ContextPipeline

## Create a Mock Retriever
In production, this would be a vector store or search API.
Here we simulate retrieval with a static document set.

In [ ]:
# Simulated document corpus
DOCS = {
    "anchor-overview": "Anchor is a context engineering toolkit for building "
        "AI applications. It provides pipelines, memory, retrieval, and agent "
        "orchestration.",
    "anchor-agents": "The Agent class wraps ContextPipeline + Anthropic SDK + "
        "tool loop. Use .with_system_prompt(), .with_memory(), and .with_tools() "
        "for configuration.",
    "anchor-memory": "Anchor memory supports sliding window, summary buffer, "
        "and graph memory strategies. MemoryManager is the unified facade.",
    "anchor-retrieval": "Retrievers fetch relevant documents at query time. "
        "Results are injected as ContextItems into the pipeline.",
    "anchor-skills": "Skills bundle tools into discoverable groups. Use "
        "always-on or on-demand activation to control tool availability.",
}

print(f"Document corpus: {len(DOCS)} documents")
for doc_id in DOCS:
    print(f"  - {doc_id}")

## Define a Search Tool
The search tool performs keyword matching against the corpus
and returns matching documents.

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for relevant documents."""
    query_lower = query.lower()
    results = []
    for doc_id, content in DOCS.items():
        if any(word in content.lower() for word in query_lower.split()):
            results.append(f"[{doc_id}] {content}")
    if not results:
        return "No documents found."
    return "\n\n".join(results[:3])

# Test the search tool
result = search_knowledge_base.fn(query="memory")
print("Search results for 'memory':")
print(result)

## Build Context Items from Retrieved Documents
Show how retrieved content becomes `ContextItem` objects
that feed into the pipeline.

In [ ]:
# Simulate what the pipeline does with retrieval results
retrieved_items = []
for doc_id, content in list(DOCS.items())[:2]:
    item = ContextItem(
        content=content,
        source_type=SourceType.RETRIEVAL,
        metadata={"doc_id": doc_id},
        priority=5,
    )
    retrieved_items.append(item)

print(f"Context items created: {len(retrieved_items)}")
for item in retrieved_items:
    print(f"  [{item.source_type.value}] {item.metadata['doc_id']}: "
          f"{str(item.content)[:50]}...")

## Wire Agent with RAG Skill
Create a skill that bundles the search tool, then attach
it to an Agent.

In [ ]:
rag_skill = Skill(
    name="knowledge_base",
    description="Search the internal knowledge base",
    instructions="Use search_knowledge_base to find relevant documents "
                "before answering questions about Anchor.",
    tools=(search_knowledge_base,),
    activation="always",
)

print(f"RAG skill: {rag_skill.name}")
print(f"  activation: {rag_skill.activation}")
print(f"  tools: {[t.name for t in rag_skill.tools]}")

## Full Agent Configuration
Here is the complete pattern for an agent with retrieval.
Note: `agent.chat()` requires an API key, so we show the
configuration without executing a live call.

In [ ]:
# Full configuration pattern (shown without live API call)
agent = Agent(
    model="claude-haiku-4-5-20251001",
    max_response_tokens=2048,
    max_rounds=10,
)

memory = SlidingWindowMemory(max_tokens=4096)

agent.with_system_prompt(
    "You are a helpful assistant with access to a knowledge base. "
    "Always search for relevant documents before answering."
)
agent.with_memory(memory)
agent.with_skill(rag_skill)

print("Agent configured with:")
print(f"  System prompt: set")
print(f"  Memory: {type(memory).__name__} ({memory.max_tokens} tokens)")
print(f"  Skills: knowledge_base")
print()
print("To run a chat loop:")
print('  for chunk in agent.chat("What is Anchor?"):')
print('      print(chunk, end="", flush=True)')

## Agent Tool Loop (Conceptual)
When `agent.chat()` runs, it follows this loop:

1. Build context (pipeline + memory + retrieval items)
2. Send to Anthropic API with tool definitions
3. If model calls a tool -> execute it, feed result back
4. Repeat until model produces final text or max_rounds reached
5. Stream text chunks to the caller

In [ ]:
# Simulate the tool loop conceptually
print("=== Simulated Agent Tool Loop ===")
print()
print("Round 1:")
print("  User: 'What memory strategies does Anchor support?'")
print("  Agent calls: search_knowledge_base(query='memory strategies')")

search_result = search_knowledge_base.fn(query="memory strategies")
print(f"  Tool result: {search_result[:80]}...")
print()
print("Round 2:")
print("  Agent receives tool result and generates final answer.")
print("  -> 'Anchor supports sliding window, summary buffer, and graph memory.'")
print()
print("Total rounds: 2 (1 tool call + 1 final response)")

## Key Takeaways
- Wrap retrieval logic in `@tool` decorated functions
- Bundle search tools into a `Skill` with `activation="always"`
- Attach skills to the Agent via `.with_skill()`
- Retrieved documents become `ContextItem` objects in the pipeline
- The agent's tool loop automatically handles search -> synthesize flows
- Use `max_rounds` to cap the number of tool-use iterations

**Next:** [Streaming Agent](05_streaming_agent.ipynb)